# 06 — Generate Initial Payment History

## Purpose

This notebook generates a deterministic payment-gateway history from the validated initial invoice dataset.

The dataset models successful payments, failed collection attempts, retry attempts, and pending transactions. It provides the financial evidence required to reconcile billing-system invoice balances with payment-gateway settlements.

## Business Grain

One row represents one payment attempt for one invoice.

An invoice may have multiple payment attempts, but each payment attempt has a unique payment and provider transaction identifier.

## Payment Scenarios

- One successful payment for every Paid invoice
- Prior failed attempts for a deterministic subset of Paid invoices
- Failed collection attempts for every Past Due invoice
- Second retry attempts for a deterministic subset of Past Due invoices
- Pending attempts for a deterministic subset of Open invoices
- No payment attempts for Voided invoices

## Data Quality Controls

- Required-field validation
- Payment identifier uniqueness
- Provider transaction identifier uniqueness
- Invoice, subscription, and customer referential integrity
- Invoice/payment ownership validation
- Payment-attempt sequence validation
- Transaction currency and amount validation
- Payment and invoice temporal consistency
- Successful settlement reconciliation
- Failed and pending payment-state validation
- Paid-invoice coverage validation
- Non-paid invoice settlement prevention
- Post-write validation

## Input

- `/Volumes/workspace/revenue_leakage_bronze/landing/billing_system/invoices/initial_load`

## Target

- `/Volumes/workspace/revenue_leakage_bronze/landing/payment_gateway/payments/initial_load`

## 1. Configuration and Schemas

Define deterministic payment-generation parameters, source and target locations, and explicit schemas for invoice and payment records.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    TimestampType,
    IntegerType,
    DecimalType,
)

EXPECTED_INVOICE_COUNT = 26925

PAID_RETRY_PERCENT = 15
PAST_DUE_SECOND_RETRY_PERCENT = 25
OPEN_PENDING_PERCENT = 30

SNAPSHOT_DATE = "2026-08-15"

LANDING_PATH = (
    "/Volumes/workspace/"
    "revenue_leakage_bronze/landing"
)

INVOICES_INITIAL_PATH = (
    f"{LANDING_PATH}/billing_system/"
    "invoices/initial_load"
)

PAYMENTS_INITIAL_PATH = (
    f"{LANDING_PATH}/payment_gateway/"
    "payments/initial_load"
)

INVOICE_SCHEMA = StructType([
    StructField(
        "invoice_id",
        StringType(),
        False
    ),
    StructField(
        "subscription_id",
        StringType(),
        False
    ),
    StructField(
        "customer_id",
        StringType(),
        False
    ),
    StructField(
        "billing_period_start",
        DateType(),
        False
    ),
    StructField(
        "billing_period_end",
        DateType(),
        False
    ),
    StructField(
        "invoice_date",
        DateType(),
        False
    ),
    StructField(
        "due_date",
        DateType(),
        False
    ),
    StructField(
        "billing_frequency",
        StringType(),
        False
    ),
    StructField(
        "currency",
        StringType(),
        False
    ),
    StructField(
        "list_price_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "discount_percentage",
        DecimalType(5, 2),
        False
    ),
    StructField(
        "discount_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "net_subscription_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "overage_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "subtotal_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "tax_rate",
        DecimalType(5, 2),
        False
    ),
    StructField(
        "tax_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "invoice_total_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "amount_paid",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "outstanding_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "voided_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "invoice_status",
        StringType(),
        False
    ),
    StructField(
        "payment_terms_days",
        IntegerType(),
        False
    ),
    StructField(
        "operation",
        StringType(),
        False
    ),
    StructField(
        "event_timestamp",
        TimestampType(),
        False
    ),
    StructField(
        "snapshot_date",
        DateType(),
        False
    ),
])

PAYMENT_SCHEMA = StructType([
    StructField(
        "payment_id",
        StringType(),
        False
    ),
    StructField(
        "invoice_id",
        StringType(),
        False
    ),
    StructField(
        "subscription_id",
        StringType(),
        False
    ),
    StructField(
        "customer_id",
        StringType(),
        False
    ),
    StructField(
        "attempt_number",
        IntegerType(),
        False
    ),
    StructField(
        "payment_status",
        StringType(),
        False
    ),
    StructField(
        "payment_method",
        StringType(),
        False
    ),
    StructField(
        "payment_provider",
        StringType(),
        False
    ),
    StructField(
        "provider_transaction_id",
        StringType(),
        False
    ),
    StructField(
        "transaction_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "settled_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "currency",
        StringType(),
        False
    ),
    StructField(
        "payment_date",
        DateType(),
        False
    ),
    StructField(
        "settlement_date",
        DateType(),
        True
    ),
    StructField(
        "failure_reason",
        StringType(),
        True
    ),
    StructField(
        "operation",
        StringType(),
        False
    ),
    StructField(
        "event_timestamp",
        TimestampType(),
        False
    ),
    StructField(
        "snapshot_date",
        DateType(),
        False
    ),
])

PAYMENT_COLUMNS = PAYMENT_SCHEMA.fieldNames()

## 2. Load and Validate the Invoice Source

Load the persisted invoice history using its explicit schema.

Validate the upstream invoice contract before generating payment attempts. The controls confirm row-count preservation, invoice uniqueness, required fields, supported lifecycle states, currency consistency, and financial balance reconciliation.

In [0]:
invoices_initial_df = (
    spark.read
    .schema(INVOICE_SCHEMA)
    .json(INVOICES_INITIAL_PATH)
)

invoice_count = (
    invoices_initial_df.count()
)

distinct_invoice_count = (
    invoices_initial_df
    .select("invoice_id")
    .distinct()
    .count()
)

invoice_null_required_condition = F.lit(False)

for column_name in INVOICE_SCHEMA.fieldNames():
    invoice_null_required_condition = (
        invoice_null_required_condition
        | F.col(column_name).isNull()
    )

invoice_null_required_field_count = (
    invoices_initial_df
    .filter(invoice_null_required_condition)
    .count()
)

invalid_invoice_status_count = (
    invoices_initial_df
    .filter(
        ~F.col("invoice_status").isin(
            "Paid",
            "Open",
            "Past Due",
            "Voided"
        )
    )
    .count()
)

invalid_invoice_currency_count = (
    invoices_initial_df
    .filter(
        F.col("currency") != "USD"
    )
    .count()
)

invoice_balance_error_count = (
    invoices_initial_df
    .filter(
        F.abs(
            F.col("invoice_total_amount")
            - (
                F.col("amount_paid")
                + F.col("outstanding_amount")
                + F.col("voided_amount")
            )
        ) > 0.01
    )
    .count()
)

invoice_status_summary = (
    invoices_initial_df
    .agg(
        F.sum(
            F.when(
                F.col("invoice_status") == "Paid",
                1
            ).otherwise(0)
        ).alias("paid_invoice_count"),
        F.sum(
            F.when(
                F.col("invoice_status") == "Open",
                1
            ).otherwise(0)
        ).alias("open_invoice_count"),
        F.sum(
            F.when(
                F.col("invoice_status") == "Past Due",
                1
            ).otherwise(0)
        ).alias("past_due_invoice_count"),
        F.sum(
            F.when(
                F.col("invoice_status") == "Voided",
                1
            ).otherwise(0)
        ).alias("voided_invoice_count")
    )
    .first()
)

paid_invoice_count = (
    invoice_status_summary["paid_invoice_count"]
)

open_invoice_count = (
    invoice_status_summary["open_invoice_count"]
)

past_due_invoice_count = (
    invoice_status_summary["past_due_invoice_count"]
)

voided_invoice_count = (
    invoice_status_summary["voided_invoice_count"]
)

assert invoice_count == EXPECTED_INVOICE_COUNT, (
    f"Expected {EXPECTED_INVOICE_COUNT:,} invoices, "
    f"but found {invoice_count:,}."
)

assert distinct_invoice_count == invoice_count, (
    "Duplicate invoice identifiers were detected."
)

assert invoice_null_required_field_count == 0, (
    "Null values were detected in required invoice fields."
)

assert invalid_invoice_status_count == 0, (
    "Unsupported invoice lifecycle states were detected."
)

assert invalid_invoice_currency_count == 0, (
    "Unsupported invoice currencies were detected."
)

assert invoice_balance_error_count == 0, (
    "Upstream invoice balance errors were detected."
)

assert (
    paid_invoice_count
    + open_invoice_count
    + past_due_invoice_count
    + voided_invoice_count
    == invoice_count
), "Invoice status counts do not reconcile."

print(
    f"Invoices loaded: "
    f"{invoice_count:,}"
)

print(
    f"Distinct invoice IDs: "
    f"{distinct_invoice_count:,}"
)

print(
    f"Null required fields: "
    f"{invoice_null_required_field_count:,}"
)

print(
    f"Invalid invoice statuses: "
    f"{invalid_invoice_status_count:,}"
)

print(
    f"Invalid invoice currencies: "
    f"{invalid_invoice_currency_count:,}"
)

print(
    f"Invoice balance errors: "
    f"{invoice_balance_error_count:,}"
)

print(
    f"Paid invoices: "
    f"{paid_invoice_count:,}"
)

print(
    f"Open invoices: "
    f"{open_invoice_count:,}"
)

print(
    f"Past Due invoices: "
    f"{past_due_invoice_count:,}"
)

print(
    f"Voided invoices: "
    f"{voided_invoice_count:,}"
)

display(
    invoices_initial_df
    .groupBy("invoice_status")
    .agg(
        F.count("*").alias("invoice_count"),
        F.round(
            F.sum("invoice_total_amount"),
            2
        ).alias("invoice_total_amount")
    )
    .orderBy("invoice_status")
)

## 3. Generate Deterministic Payment Attempts

Generate payment-gateway events from invoice lifecycle states.

Every Paid invoice receives one successful settlement. Deterministic subsets receive prior failed attempts, Past Due invoices receive failed collection attempts and optional retries, and eligible Open invoices receive pending attempts. Voided invoices generate no payment activity.

In [0]:
payment_generation_base_df = (
    invoices_initial_df
    .select(
        "invoice_id",
        "subscription_id",
        "customer_id",
        "invoice_date",
        "due_date",
        "invoice_total_amount",
        "invoice_status",
        "currency",
    )
    .withColumn(
        "paid_retry_bucket",
        F.pmod(
            F.xxhash64(
                F.concat(
                    F.col("invoice_id"),
                    F.lit("-paid-retry")
                )
            ),
            F.lit(100)
        )
    )
    .withColumn(
        "past_due_retry_bucket",
        F.pmod(
            F.xxhash64(
                F.concat(
                    F.col("invoice_id"),
                    F.lit("-past-due-retry")
                )
            ),
            F.lit(100)
        )
    )
    .withColumn(
        "open_pending_bucket",
        F.pmod(
            F.xxhash64(
                F.concat(
                    F.col("invoice_id"),
                    F.lit("-open-pending")
                )
            ),
            F.lit(100)
        )
    )
    .withColumn(
        "payment_method_bucket",
        F.pmod(
            F.xxhash64(
                F.concat(
                    F.col("customer_id"),
                    F.lit("-method")
                )
            ),
            F.lit(100)
        )
    )
    .withColumn(
        "provider_bucket",
        F.pmod(
            F.xxhash64(
                F.concat(
                    F.col("customer_id"),
                    F.lit("-provider")
                )
            ),
            F.lit(100)
        )
    )
    .withColumn(
        "payment_method",
        F.when(
            F.col("payment_method_bucket") < 70,
            "Credit Card"
        )
        .when(
            F.col("payment_method_bucket") < 90,
            "ACH"
        )
        .otherwise("Bank Transfer")
    )
    .withColumn(
        "payment_provider",
        F.when(
            F.col("payment_method") == "ACH",
            "Stripe"
        )
        .when(
            F.col("payment_method") == "Bank Transfer",
            "Adyen"
        )
        .when(
            F.col("provider_bucket") < 55,
            "Stripe"
        )
        .otherwise("Adyen")
    )
    .withColumn(
        "failure_reason_value",
        F.when(
            F.col("payment_method") == "ACH",
            "Insufficient Funds"
        )
        .when(
            F.col("payment_method") == "Bank Transfer",
            "Bank Rejected"
        )
        .when(
            F.col("provider_bucket") < 70,
            "Card Declined"
        )
        .otherwise("Processor Timeout")
    )
)

paid_successful_attempts_df = (
    payment_generation_base_df
    .filter(
        F.col("invoice_status") == "Paid"
    )
    .withColumn(
        "success_day_offset",
        F.pmod(
            F.xxhash64(
                F.concat(
                    F.col("invoice_id"),
                    F.lit("-success-day")
                )
            ),
            F.lit(4)
        ).cast("int")
    )
    .withColumn(
        "payment_date",
        F.least(
            F.lit(SNAPSHOT_DATE).cast("date"),
            F.date_add(
                F.col("invoice_date"),
                F.col("success_day_offset")
            )
        )
    )
    .withColumn(
        "attempt_number",
        F.when(
            F.col("paid_retry_bucket")
            < PAID_RETRY_PERCENT,
            2
        ).otherwise(1)
    )
    .withColumn(
        "payment_status",
        F.lit("Succeeded")
    )
    .withColumn(
        "transaction_amount",
        F.col("invoice_total_amount")
    )
    .withColumn(
        "settled_amount",
        F.col("invoice_total_amount")
    )
    .withColumn(
        "settlement_date",
        F.least(
            F.lit(SNAPSHOT_DATE).cast("date"),
            F.date_add(
                F.col("payment_date"),
                1
            )
        )
    )
    .withColumn(
        "failure_reason",
        F.lit(None).cast("string")
    )
)

paid_failed_attempts_df = (
    payment_generation_base_df
    .filter(
        (F.col("invoice_status") == "Paid")
        & (
            F.col("paid_retry_bucket")
            < PAID_RETRY_PERCENT
        )
    )
    .withColumn(
        "success_day_offset",
        F.pmod(
            F.xxhash64(
                F.concat(
                    F.col("invoice_id"),
                    F.lit("-success-day")
                )
            ),
            F.lit(4)
        ).cast("int")
    )
    .withColumn(
        "success_payment_date",
        F.least(
            F.lit(SNAPSHOT_DATE).cast("date"),
            F.date_add(
                F.col("invoice_date"),
                F.col("success_day_offset")
            )
        )
    )
    .withColumn(
        "payment_date",
        F.greatest(
            F.col("invoice_date"),
            F.date_sub(
                F.col("success_payment_date"),
                1
            )
        )
    )
    .withColumn(
        "attempt_number",
        F.lit(1)
    )
    .withColumn(
        "payment_status",
        F.lit("Failed")
    )
    .withColumn(
        "transaction_amount",
        F.col("invoice_total_amount")
    )
    .withColumn(
        "settled_amount",
        F.lit(0.00).cast(DecimalType(12, 2))
    )
    .withColumn(
        "settlement_date",
        F.lit(None).cast("date")
    )
    .withColumn(
        "failure_reason",
        F.col("failure_reason_value")
    )
)

past_due_first_attempts_df = (
    payment_generation_base_df
    .filter(
        F.col("invoice_status") == "Past Due"
    )
    .withColumn(
        "attempt_number",
        F.lit(1)
    )
    .withColumn(
        "payment_status",
        F.lit("Failed")
    )
    .withColumn(
        "transaction_amount",
        F.col("invoice_total_amount")
    )
    .withColumn(
        "settled_amount",
        F.lit(0.00).cast(DecimalType(12, 2))
    )
    .withColumn(
        "payment_date",
        F.col("due_date")
    )
    .withColumn(
        "settlement_date",
        F.lit(None).cast("date")
    )
    .withColumn(
        "failure_reason",
        F.col("failure_reason_value")
    )
)

past_due_second_attempts_df = (
    payment_generation_base_df
    .filter(
        (F.col("invoice_status") == "Past Due")
        & (
            F.col("past_due_retry_bucket")
            < PAST_DUE_SECOND_RETRY_PERCENT
        )
    )
    .withColumn(
        "attempt_number",
        F.lit(2)
    )
    .withColumn(
        "payment_status",
        F.lit("Failed")
    )
    .withColumn(
        "transaction_amount",
        F.col("invoice_total_amount")
    )
    .withColumn(
        "settled_amount",
        F.lit(0.00).cast(DecimalType(12, 2))
    )
    .withColumn(
        "payment_date",
        F.least(
            F.lit(SNAPSHOT_DATE).cast("date"),
            F.date_add(
                F.col("due_date"),
                5
            )
        )
    )
    .withColumn(
        "settlement_date",
        F.lit(None).cast("date")
    )
    .withColumn(
        "failure_reason",
        F.col("failure_reason_value")
    )
)

open_pending_attempts_df = (
    payment_generation_base_df
    .filter(
        (F.col("invoice_status") == "Open")
        & (
            F.col("open_pending_bucket")
            < OPEN_PENDING_PERCENT
        )
    )
    .withColumn(
        "attempt_number",
        F.lit(1)
    )
    .withColumn(
        "payment_status",
        F.lit("Pending")
    )
    .withColumn(
        "transaction_amount",
        F.col("invoice_total_amount")
    )
    .withColumn(
        "settled_amount",
        F.lit(0.00).cast(DecimalType(12, 2))
    )
    .withColumn(
        "payment_date",
        F.least(
            F.lit(SNAPSHOT_DATE).cast("date"),
            F.date_add(
                F.col("invoice_date"),
                1
            )
        )
    )
    .withColumn(
        "settlement_date",
        F.lit(None).cast("date")
    )
    .withColumn(
        "failure_reason",
        F.lit(None).cast("string")
    )
)

PAYMENT_ATTEMPT_COLUMNS = [
    "invoice_id",
    "subscription_id",
    "customer_id",
    "attempt_number",
    "payment_status",
    "payment_method",
    "payment_provider",
    "transaction_amount",
    "settled_amount",
    "currency",
    "payment_date",
    "settlement_date",
    "failure_reason",
]

payment_attempts_df = (
    paid_successful_attempts_df
    .select(*PAYMENT_ATTEMPT_COLUMNS)
    .unionByName(
        paid_failed_attempts_df
        .select(*PAYMENT_ATTEMPT_COLUMNS)
    )
    .unionByName(
        past_due_first_attempts_df
        .select(*PAYMENT_ATTEMPT_COLUMNS)
    )
    .unionByName(
        past_due_second_attempts_df
        .select(*PAYMENT_ATTEMPT_COLUMNS)
    )
    .unionByName(
        open_pending_attempts_df
        .select(*PAYMENT_ATTEMPT_COLUMNS)
    )
    .withColumn(
        "payment_id",
        F.concat(
            F.lit("PAY-"),
            F.col("invoice_id"),
            F.lit("-A"),
            F.lpad(
                F.col("attempt_number").cast("string"),
                2,
                "0"
            )
        )
    )
    .withColumn(
        "provider_transaction_id",
        F.concat(
            F.lit("TXN-"),
            F.upper(
                F.substring(
                    F.sha2(
                        F.concat_ws(
                            "|",
                            F.col("invoice_id"),
                            F.col("attempt_number")
                        ),
                        256
                    ),
                    1,
                    24
                )
            )
        )
    )
    .withColumn(
        "operation",
        F.lit("INSERT")
    )
    .withColumn(
        "event_time",
        F.when(
            F.col("payment_status") == "Succeeded",
            "15:00:00"
        )
        .when(
            F.col("payment_status") == "Pending",
            "11:00:00"
        )
        .when(
            F.col("attempt_number") == 1,
            "09:00:00"
        )
        .otherwise("13:00:00")
    )
    .withColumn(
        "event_timestamp",
        F.to_timestamp(
            F.concat(
                F.date_format(
                    F.col("payment_date"),
                    "yyyy-MM-dd"
                ),
                F.lit(" "),
                F.col("event_time")
            )
        )
    )
    .withColumn(
        "snapshot_date",
        F.lit(SNAPSHOT_DATE).cast("date")
    )
)

payments_initial_df = (
    payment_attempts_df
    .select(*PAYMENT_COLUMNS)
)

paid_success_count = (
    paid_successful_attempts_df.count()
)

paid_retry_failure_count = (
    paid_failed_attempts_df.count()
)

past_due_first_attempt_count = (
    past_due_first_attempts_df.count()
)

past_due_second_attempt_count = (
    past_due_second_attempts_df.count()
)

open_pending_attempt_count = (
    open_pending_attempts_df.count()
)

generated_payment_count = (
    payments_initial_df.count()
)

distinct_payment_count = (
    payments_initial_df
    .select("payment_id")
    .distinct()
    .count()
)

expected_payment_count = (
    paid_success_count
    + paid_retry_failure_count
    + past_due_first_attempt_count
    + past_due_second_attempt_count
    + open_pending_attempt_count
)

assert paid_success_count == paid_invoice_count, (
    "Not every Paid invoice received a successful payment."
)

assert past_due_first_attempt_count == past_due_invoice_count, (
    "Not every Past Due invoice received a failed attempt."
)

assert generated_payment_count == expected_payment_count, (
    "Generated payment-attempt counts do not reconcile."
)

assert distinct_payment_count == generated_payment_count, (
    "Duplicate payment identifiers were generated."
)

assert generated_payment_count > 0, (
    "The generated payment dataset is empty."
)

print(
    f"Paid successful attempts: "
    f"{paid_success_count:,}"
)

print(
    f"Paid prior failed attempts: "
    f"{paid_retry_failure_count:,}"
)

print(
    f"Past Due first attempts: "
    f"{past_due_first_attempt_count:,}"
)

print(
    f"Past Due second attempts: "
    f"{past_due_second_attempt_count:,}"
)

print(
    f"Open pending attempts: "
    f"{open_pending_attempt_count:,}"
)

print(
    f"Generated payment attempts: "
    f"{generated_payment_count:,}"
)

print(
    f"Distinct payment IDs: "
    f"{distinct_payment_count:,}"
)

display(
    payments_initial_df
    .groupBy(
        "payment_status",
        "attempt_number"
    )
    .count()
    .orderBy(
        "payment_status",
        "attempt_number"
    )
)

## 4. Validate Payment Data Quality and Financial Reconciliation

Apply fail-fast controls across payment identity, invoice ownership, attempt sequencing, lifecycle state, transaction amounts, settlement balances, and event chronology.

The validation confirms that every Paid invoice has exactly one successful settlement, Past Due invoices have failed collection evidence, non-paid invoices have no settled funds, and Voided invoices have no payment activity.

In [0]:
actual_payment_count = (
    payments_initial_df.count()
)

distinct_payment_count = (
    payments_initial_df
    .select("payment_id")
    .distinct()
    .count()
)

distinct_provider_transaction_count = (
    payments_initial_df
    .select("provider_transaction_id")
    .distinct()
    .count()
)

PAYMENT_REQUIRED_COLUMNS = [
    column_name
    for column_name in PAYMENT_COLUMNS
    if column_name not in [
        "settlement_date",
        "failure_reason",
    ]
]

null_required_condition = F.lit(False)

for column_name in PAYMENT_REQUIRED_COLUMNS:
    null_required_condition = (
        null_required_condition
        | F.col(column_name).isNull()
    )

null_required_field_count = (
    payments_initial_df
    .filter(null_required_condition)
    .count()
)

duplicate_attempt_count = (
    payments_initial_df
    .groupBy(
        "invoice_id",
        "attempt_number"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

orphan_invoice_count = (
    payments_initial_df
    .select("invoice_id")
    .distinct()
    .join(
        invoices_initial_df
        .select("invoice_id"),
        on="invoice_id",
        how="left_anti"
    )
    .count()
)

payment_ownership_mismatch_count = (
    payments_initial_df
    .alias("payment")
    .join(
        invoices_initial_df
        .select(
            "invoice_id",
            "subscription_id",
            "customer_id"
        )
        .alias("invoice"),
        on=(
            F.col("payment.invoice_id")
            == F.col("invoice.invoice_id")
        ),
        how="inner"
    )
    .filter(
        (
            F.col("payment.subscription_id")
            != F.col("invoice.subscription_id")
        )
        | (
            F.col("payment.customer_id")
            != F.col("invoice.customer_id")
        )
    )
    .count()
)

invalid_transaction_count = (
    payments_initial_df
    .alias("payment")
    .join(
        invoices_initial_df
        .select(
            "invoice_id",
            "invoice_total_amount",
            "currency"
        )
        .alias("invoice"),
        on=(
            F.col("payment.invoice_id")
            == F.col("invoice.invoice_id")
        ),
        how="inner"
    )
    .filter(
        (
            F.abs(
                F.col("payment.transaction_amount")
                - F.col("invoice.invoice_total_amount")
            ) > 0.01
        )
        | (
            F.col("payment.currency")
            != F.col("invoice.currency")
        )
        | (
            F.col("payment.transaction_amount") <= 0
        )
    )
    .count()
)

invalid_payment_state_count = (
    payments_initial_df
    .filter(
        (
            (F.col("payment_status") == "Succeeded")
            & (
                (
                    F.abs(
                        F.col("settled_amount")
                        - F.col("transaction_amount")
                    ) > 0.01
                )
                | F.col("settlement_date").isNull()
                | F.col("failure_reason").isNotNull()
            )
        )
        | (
            (F.col("payment_status") == "Failed")
            & (
                (F.col("settled_amount") != 0)
                | F.col("settlement_date").isNotNull()
                | F.col("failure_reason").isNull()
            )
        )
        | (
            (F.col("payment_status") == "Pending")
            & (
                (F.col("settled_amount") != 0)
                | F.col("settlement_date").isNotNull()
                | F.col("failure_reason").isNotNull()
            )
        )
    )
    .count()
)

invalid_payment_date_count = (
    payments_initial_df
    .alias("payment")
    .join(
        invoices_initial_df
        .select(
            "invoice_id",
            "invoice_date"
        )
        .alias("invoice"),
        on=(
            F.col("payment.invoice_id")
            == F.col("invoice.invoice_id")
        ),
        how="inner"
    )
    .filter(
        (
            F.col("payment.payment_date")
            < F.col("invoice.invoice_date")
        )
        | (
            F.col("payment.payment_date")
            > F.col("payment.snapshot_date")
        )
        | (
            F.col("payment.settlement_date").isNotNull()
            & (
                F.col("payment.settlement_date")
                < F.col("payment.payment_date")
            )
        )
        | (
            F.col("payment.settlement_date").isNotNull()
            & (
                F.col("payment.settlement_date")
                > F.col("payment.snapshot_date")
            )
        )
        | (
            F.col("payment.event_timestamp").cast("date")
            != F.col("payment.payment_date")
        )
    )
    .count()
)

attempt_sequence_summary_df = (
    payments_initial_df
    .groupBy("invoice_id")
    .agg(
        F.min("attempt_number")
        .alias("minimum_attempt_number"),

        F.max("attempt_number")
        .alias("maximum_attempt_number"),

        F.countDistinct("attempt_number")
        .alias("distinct_attempt_count"),

        F.min(
            F.when(
                F.col("attempt_number") == 1,
                F.col("event_timestamp")
            )
        ).alias("first_attempt_timestamp"),

        F.min(
            F.when(
                F.col("attempt_number") == 2,
                F.col("event_timestamp")
            )
        ).alias("second_attempt_timestamp")
    )
)

invalid_attempt_sequence_count = (
    attempt_sequence_summary_df
    .filter(
        (
            F.col("minimum_attempt_number") != 1
        )
        | (
            F.col("maximum_attempt_number")
            != F.col("distinct_attempt_count")
        )
        | (
            F.col("second_attempt_timestamp").isNotNull()
            & (
                F.col("first_attempt_timestamp").isNull()
                | (
                    F.col("first_attempt_timestamp")
                    > F.col("second_attempt_timestamp")
                )
            )
        )
    )
    .count()
)

invalid_retry_predecessor_count = (
    payments_initial_df
    .filter(
        F.col("attempt_number") == 2
    )
    .select("invoice_id")
    .join(
        payments_initial_df
        .filter(
            F.col("attempt_number") == 1
        )
        .select("invoice_id"),
        on="invoice_id",
        how="left_anti"
    )
    .count()
)

invalid_success_retry_count = (
    payments_initial_df
    .filter(
        (F.col("payment_status") == "Succeeded")
        & (F.col("attempt_number") == 2)
    )
    .select("invoice_id")
    .join(
        payments_initial_df
        .filter(
            (F.col("payment_status") == "Failed")
            & (F.col("attempt_number") == 1)
        )
        .select("invoice_id"),
        on="invoice_id",
        how="left_anti"
    )
    .count()
)

invalid_invoice_status_mapping_count = (
    payments_initial_df
    .alias("payment")
    .join(
        invoices_initial_df
        .select(
            "invoice_id",
            "invoice_status"
        )
        .alias("invoice"),
        on=(
            F.col("payment.invoice_id")
            == F.col("invoice.invoice_id")
        ),
        how="inner"
    )
    .filter(
        (
            (
                F.col("invoice.invoice_status") == "Paid"
            )
            & ~F.col("payment.payment_status").isin(
                "Succeeded",
                "Failed"
            )
        )
        | (
            (
                F.col("invoice.invoice_status") == "Past Due"
            )
            & (
                F.col("payment.payment_status") != "Failed"
            )
        )
        | (
            (
                F.col("invoice.invoice_status") == "Open"
            )
            & (
                F.col("payment.payment_status") != "Pending"
            )
        )
        | (
            F.col("invoice.invoice_status") == "Voided"
        )
    )
    .count()
)

paid_invoice_without_success_count = (
    invoices_initial_df
    .filter(
        F.col("invoice_status") == "Paid"
    )
    .select("invoice_id")
    .join(
        payments_initial_df
        .filter(
            F.col("payment_status") == "Succeeded"
        )
        .select("invoice_id")
        .distinct(),
        on="invoice_id",
        how="left_anti"
    )
    .count()
)

past_due_without_failure_count = (
    invoices_initial_df
    .filter(
        F.col("invoice_status") == "Past Due"
    )
    .select("invoice_id")
    .join(
        payments_initial_df
        .filter(
            (F.col("payment_status") == "Failed")
            & (F.col("attempt_number") == 1)
        )
        .select("invoice_id")
        .distinct(),
        on="invoice_id",
        how="left_anti"
    )
    .count()
)

duplicate_successful_invoice_count = (
    payments_initial_df
    .filter(
        F.col("payment_status") == "Succeeded"
    )
    .groupBy("invoice_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

non_paid_success_count = (
    payments_initial_df
    .filter(
        F.col("payment_status") == "Succeeded"
    )
    .alias("payment")
    .join(
        invoices_initial_df
        .select(
            "invoice_id",
            "invoice_status"
        )
        .alias("invoice"),
        on=(
            F.col("payment.invoice_id")
            == F.col("invoice.invoice_id")
        ),
        how="inner"
    )
    .filter(
        F.col("invoice.invoice_status") != "Paid"
    )
    .count()
)

voided_invoice_payment_count = (
    payments_initial_df
    .select("invoice_id")
    .join(
        invoices_initial_df
        .filter(
            F.col("invoice_status") == "Voided"
        )
        .select("invoice_id"),
        on="invoice_id",
        how="inner"
    )
    .count()
)

non_successful_settlement_count = (
    payments_initial_df
    .filter(
        (
            F.col("payment_status") != "Succeeded"
        )
        & (
            F.col("settled_amount") != 0
        )
    )
    .count()
)

successful_settlement_by_invoice_df = (
    payments_initial_df
    .filter(
        F.col("payment_status") == "Succeeded"
    )
    .groupBy("invoice_id")
    .agg(
        F.sum("settled_amount")
        .alias("successful_settled_amount")
    )
)

paid_settlement_mismatch_count = (
    invoices_initial_df
    .filter(
        F.col("invoice_status") == "Paid"
    )
    .select(
        "invoice_id",
        "amount_paid"
    )
    .join(
        successful_settlement_by_invoice_df,
        on="invoice_id",
        how="left"
    )
    .filter(
        F.col("successful_settled_amount").isNull()
        | (
            F.abs(
                F.col("amount_paid")
                - F.col("successful_settled_amount")
            ) > 0.01
        )
    )
    .count()
)

successful_payment_count = (
    payments_initial_df
    .filter(
        F.col("payment_status") == "Succeeded"
    )
    .count()
)

successful_settled_amount = (
    payments_initial_df
    .filter(
        F.col("payment_status") == "Succeeded"
    )
    .agg(
        F.round(
            F.sum("settled_amount"),
            2
        ).alias("settled_amount")
    )
    .first()["settled_amount"]
)

invoice_paid_amount = (
    invoices_initial_df
    .filter(
        F.col("invoice_status") == "Paid"
    )
    .agg(
        F.round(
            F.sum("amount_paid"),
            2
        ).alias("amount_paid")
    )
    .first()["amount_paid"]
)

invalid_domain_count = (
    payments_initial_df
    .filter(
        ~F.col("payment_status").isin(
            "Succeeded",
            "Failed",
            "Pending"
        )
        | ~F.col("payment_method").isin(
            "Credit Card",
            "ACH",
            "Bank Transfer"
        )
        | ~F.col("payment_provider").isin(
            "Stripe",
            "Adyen"
        )
        | ~F.col("attempt_number").isin(
            1,
            2
        )
        | (
            F.col("currency") != "USD"
        )
        | (
            F.col("operation") != "INSERT"
        )
        | (
            F.col("transaction_amount") <= 0
        )
        | (
            F.col("settled_amount") < 0
        )
    )
    .count()
)

assert actual_payment_count == generated_payment_count, (
    "Payment count changed after generation."
)

assert distinct_payment_count == actual_payment_count, (
    "Duplicate payment identifiers were detected."
)

assert distinct_provider_transaction_count == actual_payment_count, (
    "Duplicate provider transaction identifiers were detected."
)

assert null_required_field_count == 0, (
    "Null values were detected in required payment fields."
)

assert duplicate_attempt_count == 0, (
    "Duplicate invoice attempt numbers were detected."
)

assert orphan_invoice_count == 0, (
    "Payments referencing unknown invoices were detected."
)

assert payment_ownership_mismatch_count == 0, (
    "Payment ownership does not match its invoice."
)

assert invalid_transaction_count == 0, (
    "Invalid payment amounts or currencies were detected."
)

assert invalid_payment_state_count == 0, (
    "Invalid payment-state allocations were detected."
)

assert invalid_payment_date_count == 0, (
    "Invalid payment or settlement dates were detected."
)

assert invalid_attempt_sequence_count == 0, (
    "Invalid payment-attempt sequences were detected."
)

assert invalid_retry_predecessor_count == 0, (
    "Retry attempts without first attempts were detected."
)

assert invalid_success_retry_count == 0, (
    "Successful retries without prior failures were detected."
)

assert invalid_invoice_status_mapping_count == 0, (
    "Payment and invoice lifecycle states do not match."
)

assert paid_invoice_without_success_count == 0, (
    "Paid invoices without successful payments were detected."
)

assert past_due_without_failure_count == 0, (
    "Past Due invoices without failed attempts were detected."
)

assert duplicate_successful_invoice_count == 0, (
    "Invoices with multiple successful payments were detected."
)

assert non_paid_success_count == 0, (
    "Successful payments for non-paid invoices were detected."
)

assert voided_invoice_payment_count == 0, (
    "Payment attempts for Voided invoices were detected."
)

assert non_successful_settlement_count == 0, (
    "Failed or Pending payments contain settled amounts."
)

assert paid_settlement_mismatch_count == 0, (
    "Successful payments do not reconcile with Paid invoices."
)

assert successful_payment_count == paid_invoice_count, (
    "Successful payment count does not match Paid invoices."
)

assert successful_settled_amount == invoice_paid_amount, (
    "Successful settlement total does not match invoice payments."
)

assert invalid_domain_count == 0, (
    "Invalid payment domain values were detected."
)

print(
    f"Validated payment rows: "
    f"{actual_payment_count:,}"
)

print(
    f"Distinct payment IDs: "
    f"{distinct_payment_count:,}"
)

print(
    f"Distinct provider transaction IDs: "
    f"{distinct_provider_transaction_count:,}"
)

print(
    f"Null required fields: "
    f"{null_required_field_count:,}"
)

print(
    f"Duplicate attempts: "
    f"{duplicate_attempt_count:,}"
)

print(
    f"Orphan invoices: "
    f"{orphan_invoice_count:,}"
)

print(
    f"Ownership mismatches: "
    f"{payment_ownership_mismatch_count:,}"
)

print(
    f"Invalid transactions: "
    f"{invalid_transaction_count:,}"
)

print(
    f"Invalid payment states: "
    f"{invalid_payment_state_count:,}"
)

print(
    f"Invalid payment dates: "
    f"{invalid_payment_date_count:,}"
)

print(
    f"Invalid attempt sequences: "
    f"{invalid_attempt_sequence_count:,}"
)

print(
    f"Invalid retry predecessors: "
    f"{invalid_retry_predecessor_count:,}"
)

print(
    f"Invalid successful retries: "
    f"{invalid_success_retry_count:,}"
)

print(
    f"Invalid invoice/status mappings: "
    f"{invalid_invoice_status_mapping_count:,}"
)

print(
    f"Paid invoices without success: "
    f"{paid_invoice_without_success_count:,}"
)

print(
    f"Past Due invoices without failure: "
    f"{past_due_without_failure_count:,}"
)

print(
    f"Duplicate successful invoices: "
    f"{duplicate_successful_invoice_count:,}"
)

print(
    f"Non-paid successful payments: "
    f"{non_paid_success_count:,}"
)

print(
    f"Voided invoice payments: "
    f"{voided_invoice_payment_count:,}"
)

print(
    f"Non-successful settlements: "
    f"{non_successful_settlement_count:,}"
)

print(
    f"Paid settlement mismatches: "
    f"{paid_settlement_mismatch_count:,}"
)

print(
    f"Invalid domain values: "
    f"{invalid_domain_count:,}"
)

print(
    f"Successful settled amount: "
    f"{successful_settled_amount}"
)

print(
    f"Invoice paid amount: "
    f"{invoice_paid_amount}"
)

display(
    payments_initial_df
    .groupBy("payment_status")
    .agg(
        F.count("*").alias("payment_count"),
        F.round(
            F.sum("transaction_amount"),
            2
        ).alias("transaction_amount"),
        F.round(
            F.sum("settled_amount"),
            2
        ).alias("settled_amount")
    )
    .orderBy("payment_status")
)

## 5. Persist and Revalidate the Raw Payment History

Persist the validated payment-attempt history as raw JSON in the payment-gateway landing directory.

Reload the dataset using the explicit payment schema and verify complete row preservation, identifier uniqueness, required fields, lifecycle-state consistency, invoice references, payment distributions, and financial totals.

In [0]:
# The synthetic initial payment history is fully
# regenerated on every notebook run.
(
    payments_initial_df.write
    .format("json")
    .mode("overwrite")
    .save(PAYMENTS_INITIAL_PATH)
)

saved_payments_df = (
    spark.read
    .schema(PAYMENT_SCHEMA)
    .json(PAYMENTS_INITIAL_PATH)
)

saved_payment_count = (
    saved_payments_df.count()
)

saved_distinct_payment_count = (
    saved_payments_df
    .select("payment_id")
    .distinct()
    .count()
)

saved_distinct_provider_transaction_count = (
    saved_payments_df
    .select("provider_transaction_id")
    .distinct()
    .count()
)

saved_null_required_condition = F.lit(False)

for column_name in PAYMENT_REQUIRED_COLUMNS:
    saved_null_required_condition = (
        saved_null_required_condition
        | F.col(column_name).isNull()
    )

saved_null_required_field_count = (
    saved_payments_df
    .filter(saved_null_required_condition)
    .count()
)

saved_duplicate_attempt_count = (
    saved_payments_df
    .groupBy(
        "invoice_id",
        "attempt_number"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

saved_orphan_invoice_count = (
    saved_payments_df
    .select("invoice_id")
    .distinct()
    .join(
        invoices_initial_df
        .select("invoice_id"),
        on="invoice_id",
        how="left_anti"
    )
    .count()
)

saved_invalid_payment_state_count = (
    saved_payments_df
    .filter(
        (
            (F.col("payment_status") == "Succeeded")
            & (
                (
                    F.abs(
                        F.col("settled_amount")
                        - F.col("transaction_amount")
                    ) > 0.01
                )
                | F.col("settlement_date").isNull()
                | F.col("failure_reason").isNotNull()
            )
        )
        | (
            (F.col("payment_status") == "Failed")
            & (
                (F.col("settled_amount") != 0)
                | F.col("settlement_date").isNotNull()
                | F.col("failure_reason").isNull()
            )
        )
        | (
            (F.col("payment_status") == "Pending")
            & (
                (F.col("settled_amount") != 0)
                | F.col("settlement_date").isNotNull()
                | F.col("failure_reason").isNotNull()
            )
        )
    )
    .count()
)

saved_invalid_domain_count = (
    saved_payments_df
    .filter(
        ~F.col("payment_status").isin(
            "Succeeded",
            "Failed",
            "Pending"
        )
        | ~F.col("payment_method").isin(
            "Credit Card",
            "ACH",
            "Bank Transfer"
        )
        | ~F.col("payment_provider").isin(
            "Stripe",
            "Adyen"
        )
        | ~F.col("attempt_number").isin(
            1,
            2
        )
        | (
            F.col("currency") != "USD"
        )
        | (
            F.col("operation") != "INSERT"
        )
        | (
            F.col("transaction_amount") <= 0
        )
        | (
            F.col("settled_amount") < 0
        )
        | (
            F.col("payment_date")
            > F.col("snapshot_date")
        )
        | (
            F.col("event_timestamp").cast("date")
            != F.col("payment_date")
        )
    )
    .count()
)

source_status_counts_df = (
    payments_initial_df
    .groupBy(
        "payment_status",
        "attempt_number"
    )
    .count()
)

saved_status_counts_df = (
    saved_payments_df
    .groupBy(
        "payment_status",
        "attempt_number"
    )
    .count()
)

status_distribution_mismatch_count = (
    source_status_counts_df
    .exceptAll(saved_status_counts_df)
    .count()
    + saved_status_counts_df
    .exceptAll(source_status_counts_df)
    .count()
)

row_content_mismatch_count = (
    payments_initial_df
    .select(*PAYMENT_COLUMNS)
    .exceptAll(
        saved_payments_df
        .select(*PAYMENT_COLUMNS)
    )
    .count()
    + saved_payments_df
    .select(*PAYMENT_COLUMNS)
    .exceptAll(
        payments_initial_df
        .select(*PAYMENT_COLUMNS)
    )
    .count()
)

source_financial_summary = (
    payments_initial_df
    .agg(
        F.round(
            F.sum("transaction_amount"),
            2
        ).alias("transaction_amount"),

        F.round(
            F.sum("settled_amount"),
            2
        ).alias("settled_amount")
    )
    .first()
)

saved_financial_summary = (
    saved_payments_df
    .agg(
        F.round(
            F.sum("transaction_amount"),
            2
        ).alias("transaction_amount"),

        F.round(
            F.sum("settled_amount"),
            2
        ).alias("settled_amount")
    )
    .first()
)

saved_successful_settlement_by_invoice_df = (
    saved_payments_df
    .filter(
        F.col("payment_status") == "Succeeded"
    )
    .groupBy("invoice_id")
    .agg(
        F.sum("settled_amount")
        .alias("successful_settled_amount")
    )
)

saved_paid_settlement_mismatch_count = (
    invoices_initial_df
    .filter(
        F.col("invoice_status") == "Paid"
    )
    .select(
        "invoice_id",
        "amount_paid"
    )
    .join(
        saved_successful_settlement_by_invoice_df,
        on="invoice_id",
        how="left"
    )
    .filter(
        F.col("successful_settled_amount").isNull()
        | (
            F.abs(
                F.col("amount_paid")
                - F.col("successful_settled_amount")
            ) > 0.01
        )
    )
    .count()
)

saved_non_successful_settlement_count = (
    saved_payments_df
    .filter(
        (
            F.col("payment_status") != "Succeeded"
        )
        & (
            F.col("settled_amount") != 0
        )
    )
    .count()
)

saved_voided_invoice_payment_count = (
    saved_payments_df
    .select("invoice_id")
    .join(
        invoices_initial_df
        .filter(
            F.col("invoice_status") == "Voided"
        )
        .select("invoice_id"),
        on="invoice_id",
        how="inner"
    )
    .count()
)

assert saved_payment_count == actual_payment_count, (
    "Saved payment count does not match "
    "the generated dataset."
)

assert saved_distinct_payment_count == saved_payment_count, (
    "Duplicate payment identifiers were found "
    "after persistence."
)

assert (
    saved_distinct_provider_transaction_count
    == saved_payment_count
), (
    "Duplicate provider transaction identifiers "
    "were found after persistence."
)

assert saved_null_required_field_count == 0, (
    "Null required payment fields were found "
    "after persistence."
)

assert saved_duplicate_attempt_count == 0, (
    "Duplicate payment attempts were found "
    "after persistence."
)

assert saved_orphan_invoice_count == 0, (
    "Payments referencing unknown invoices were found "
    "after persistence."
)

assert saved_invalid_payment_state_count == 0, (
    "Invalid payment states were found "
    "after persistence."
)

assert saved_invalid_domain_count == 0, (
    "Invalid payment domain values were found "
    "after persistence."
)

assert status_distribution_mismatch_count == 0, (
    "Payment status distributions changed "
    "during persistence."
)

assert row_content_mismatch_count == 0, (
    "Payment row contents changed during persistence."
)

assert (
    saved_financial_summary["transaction_amount"]
    == source_financial_summary["transaction_amount"]
), "Transaction totals changed during persistence."

assert (
    saved_financial_summary["settled_amount"]
    == source_financial_summary["settled_amount"]
), "Settlement totals changed during persistence."

assert saved_paid_settlement_mismatch_count == 0, (
    "Saved settlements do not reconcile with Paid invoices."
)

assert saved_non_successful_settlement_count == 0, (
    "Saved Failed or Pending payments contain settlements."
)

assert saved_voided_invoice_payment_count == 0, (
    "Saved payment attempts exist for Voided invoices."
)

print(
    f"Saved payment rows: "
    f"{saved_payment_count:,}"
)

print(
    f"Saved distinct payment IDs: "
    f"{saved_distinct_payment_count:,}"
)

print(
    f"Saved distinct provider transaction IDs: "
    f"{saved_distinct_provider_transaction_count:,}"
)

print(
    f"Saved null required fields: "
    f"{saved_null_required_field_count:,}"
)

print(
    f"Saved duplicate attempts: "
    f"{saved_duplicate_attempt_count:,}"
)

print(
    f"Saved orphan invoices: "
    f"{saved_orphan_invoice_count:,}"
)

print(
    f"Saved invalid payment states: "
    f"{saved_invalid_payment_state_count:,}"
)

print(
    f"Saved invalid domain values: "
    f"{saved_invalid_domain_count:,}"
)

print(
    f"Status distribution mismatches: "
    f"{status_distribution_mismatch_count:,}"
)

print(
    f"Row content mismatches: "
    f"{row_content_mismatch_count:,}"
)

print(
    f"Saved paid settlement mismatches: "
    f"{saved_paid_settlement_mismatch_count:,}"
)

print(
    f"Saved non-successful settlements: "
    f"{saved_non_successful_settlement_count:,}"
)

print(
    f"Saved Voided invoice payments: "
    f"{saved_voided_invoice_payment_count:,}"
)

print(
    f"Saved transaction total: "
    f"{saved_financial_summary['transaction_amount']}"
)

print(
    f"Saved settlement total: "
    f"{saved_financial_summary['settled_amount']}"
)

print(
    f"Target path: "
    f"{PAYMENTS_INITIAL_PATH}"
)

display(
    saved_payments_df
    .groupBy(
        "payment_status",
        "attempt_number"
    )
    .count()
    .orderBy(
        "payment_status",
        "attempt_number"
    )
)